In [10]:
from pathlib import Path
import pandas as pd

results_dir = Path().resolve().parent.parent / "assets" / "results" / "tsfm"
print(results_dir.absolute())

if not results_dir.exists():
    raise FileNotFoundError(f"Results directory not found: {results_dir}")


/Users/vcerqueira/Dropbox/Research/experiments-metaarima/assets/results/tsfm


In [14]:
id_columns = {
    "unique_id",
    "id",
    "series_id",
    "timestamp",
    "time",
    "date",
    "ds",
    "y",
    "target",
    "split",
    "fold",
    "horizon",
}

rows = []

for file in sorted(results_dir.glob("*/scores*.csv")):
    df = pd.read_csv(file)

    method_columns = [
        col
        for col in df.columns
        if col not in id_columns and pd.api.types.is_numeric_dtype(df[col])
    ]

    if not method_columns:
        continue

    row = {
        "file": file.name,
        "dataset": file.stem.replace("scores,", "")
        .replace("scores_", "")
        .replace("scores-", ""),
        "n_series": len(df),
    }

    row.update(df[method_columns].mean(numeric_only=True).to_dict())
    rows.append(row)

raw_summary = pd.DataFrame(rows)

method_columns = [
    col for col in raw_summary.columns if col not in {"file", "dataset", "n_series"}
]

summary = raw_summary.groupby(["file", "dataset"], as_index=False).agg(
    {"n_series": "max", **{col: "mean" for col in method_columns}}
)

best_two = summary[method_columns].apply(
    lambda row: row.dropna().nsmallest(2).index.tolist(), axis=1
)
best_two = best_two.apply(lambda x: x + [pd.NA] * (2 - len(x)))

summary[["best_model", "second_best_model"]] = pd.DataFrame(
    best_two.tolist(), index=summary.index
)

summary = (
    summary[
        ["dataset", *method_columns]
    ]
    .sort_values("dataset")
    .reset_index(drop=True)
).drop(columns=['SeasonalNaive']).set_index('dataset')

summary.round(2)

,MetaARIMA,AutoARIMA,Chronos2,Moirai2,TimesFM
dataset,,,,,
monash_m1_monthly,0.88,0.92,0.87,0.97,0.90
monash_m3_monthly,0.71,0.74,0.69,0.74,0.72
monash_tourism_monthly,1.17,1.22,1.17,1.25,1.36


In [17]:
tab = summary.round(3).astype(str).to_latex(caption='cap',label='tab:tsfm')
print(tab)

\begin{table}
\caption{cap}
\label{tab:tsfm}
\begin{tabular}{llllll}
\toprule
 & MetaARIMA & AutoARIMA & Chronos2 & Moirai2 & TimesFM \\
dataset &  &  &  &  &  \\
\midrule
monash_m1_monthly & 0.876 & 0.917 & 0.867 & 0.97 & 0.899 \\
monash_m3_monthly & 0.714 & 0.737 & 0.693 & 0.738 & 0.723 \\
monash_tourism_monthly & 1.168 & 1.221 & 1.166 & 1.25 & 1.359 \\
\bottomrule
\end{tabular}
\end{table}



In [19]:
raw_summary.query('dataset=="monash_m3_monthly"')

,file,dataset,n_series,MetaARIMA,AutoARIMA,SeasonalNaive,Chronos2,Moirai2,TimesFM
1,"scores,monash_m3_monthly.csv",monash_m3_monthly,1428,0.71371,0.736934,1.001809,0.693486,NaN,NaN
4,"scores,monash_m3_monthly.csv",monash_m3_monthly,1428,0.71371,0.736934,1.001809,NaN,0.737527,NaN
7,"scores,monash_m3_monthly.csv",monash_m3_monthly,1428,0.71371,0.736934,1.001809,NaN,NaN,0.723489


In [23]:
dataset = "monash_m3_monthly"
m3_scores = pd.read_csv(next(results_dir.glob(f"chronos2/scores*{dataset}*.csv")))

chronos2_best_id = m3_scores.loc[m3_scores["Chronos2"].idxmin(), "unique_id"]
metaarima_best_id = m3_scores.loc[m3_scores["MetaARIMA"].idxmin(), "unique_id"]

pd.Series(
    {
        "Chronos2": chronos2_best_id,
        "MetaARIMA": metaarima_best_id,
    },
    name=dataset,
)

Chronos2     T001407
MetaARIMA    T000816
Name: monash_m3_monthly, dtype: object

In [24]:
m3_scores["diff"] = m3_scores["Chronos2"] - m3_scores["MetaARIMA"]

largest_diff = m3_scores.loc[m3_scores["diff"].idxmax()]
smallest_diff = m3_scores.loc[m3_scores["diff"].idxmin()]

pd.DataFrame(
    {
        "unique_id": [largest_diff["unique_id"], smallest_diff["unique_id"]],
        "Chronos2": [largest_diff["Chronos2"], smallest_diff["Chronos2"]],
        "MetaARIMA": [largest_diff["MetaARIMA"], smallest_diff["MetaARIMA"]],
        "Chronos2 - MetaARIMA": [largest_diff["diff"], smallest_diff["diff"]],
    },
    index=["MetaARIMA better", "Chronos2 better"],
)

,unique_id,Chronos2,MetaARIMA,Chronos2 - MetaARIMA
MetaARIMA better,T001384,4.922456,1.454163,3.468293
Chronos2 better,T000696,0.601883,6.357008,-5.755125
